# 04_207 · Comparación final de modelos con cuatro categorías

Este cuaderno no entrena. Reúne únicamente evaluaciones realizadas sobre validation/test 4:1 comunes y verifica que todos los resultados declaren el mismo hash de dataset. Debe ejecutarse después de los experimentos que se quieran comparar. Al comenzar en Windows, comprueba el bundle `G:\My Drive\PLN_colab_04_artifacts` y recupera hacia el workspace las métricas faltantes producidas en Colab; los archivos locales diferentes nunca se sobrescriben silenciosamente. Qwen plano se carga desde la selección operativa de validation (época 3), y cualquier comparación jerárquica antigua se recalcula contra ese mismo checkpoint sin reentrenar.

In [ ]:
from pathlib import Path
import hashlib, importlib, json, subprocess, sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
ROOT = Path.cwd().resolve()
if ROOT.name.lower() == 'cuadernos': ROOT = ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))

# Recuperación automática y segura de resultados producidos en Colab.
DRIVE_BUNDLE = Path(r'G:\My Drive\PLN_colab_04_artifacts')
RECOVERY_SCRIPT = ROOT / 'scripts_auxiliares' / 'recuperar_resultados_colab_04_20x.ps1'
COLAB_RESULT_PATHS = {
    '04_205 Qwen plano operativo': Path('resultados/metricas/qwen3_06b_lora_acoso_amenaza_4/evaluacion_test_modelo_seleccionado.json'),
    '04_202 Transformer plano': Path('resultados/metricas/transformer_plano_4/resultado.json'),
    '04_203 Transformer en cascada': Path('resultados/metricas/experimentos_jerarquicos_4/cascada_binaria_multietiqueta_4_seguros_ampliados/resultado.json'),
    '04_204 Transformer jerárquico': Path('resultados/metricas/experimentos_jerarquicos_4/transformer_jerarquico_multitarea_4/resultado.json'),
    '04_206 Qwen jerárquico': Path('resultados/metricas/qwen_jerarquico_4/resultado.json'),
}

def _sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def _sync_state(relative):
    local, remote = ROOT / relative, DRIVE_BUNDLE / relative
    if local.is_file() and remote.is_file():
        return 'idéntico' if _sha256(local) == _sha256(remote) else 'conflicto'
    if local.is_file(): return 'solo local'
    if remote.is_file(): return 'solo Drive'
    return 'ausente'

states_before = {name: _sync_state(path) for name, path in COLAB_RESULT_PATHS.items()}
# 04_206 puede haberse recalculado localmente contra la época operativa sin reentrenar.
for name, relative in COLAB_RESULT_PATHS.items():
    if name.startswith('04_206') and states_before[name] == 'conflicto':
        local_result, drive_result = json.loads((ROOT / relative).read_text(encoding='utf-8')), json.loads((DRIVE_BUNDLE / relative).read_text(encoding='utf-8'))
        if local_result.get('qwen_flat_reference') and not drive_result.get('qwen_flat_reference'):
            states_before[name] = 'local actualizado'
conflicts = [name for name, state in states_before.items() if state == 'conflicto']
if conflicts:
    raise RuntimeError('Resultados distintos en Drive y D: ' + ', '.join(conflicts) + '. Revíselos antes de usar -Force.')
recoverable = [name for name, state in states_before.items() if state == 'solo Drive']
if recoverable:
    if sys.platform != 'win32' or not RECOVERY_SCRIPT.is_file():
        raise RuntimeError('Hay resultados sólo en Drive, pero la recuperación automática requiere Windows y el script local.')
    common = ['powershell', '-NoProfile', '-ExecutionPolicy', 'Bypass', '-File', str(RECOVERY_SCRIPT),
              '-Workspace', str(ROOT), '-DriveBundle', str(DRIVE_BUNDLE)]
    modes = []
    if any(name.startswith(('04_202', '04_203', '04_204')) for name in recoverable):
        modes.append('-ComparisonOnly')
    if any(name.startswith(('04_205', '04_206')) for name in recoverable):
        modes.append('-Qwen04_20XOnly')
    for mode in modes:
        recovery = subprocess.run([*common, mode], text=True, capture_output=True)
        if recovery.returncode != 0:
            raise RuntimeError(f'Falló la recuperación {mode} desde Drive:\n' + (recovery.stderr or recovery.stdout))
        print(recovery.stdout.strip())
elif not DRIVE_BUNDLE.is_dir():
    print(f'Drive no está disponible en {DRIVE_BUNDLE}; se usarán sólo resultados locales.')

DEPLOYMENT_REQUIRED = [
    Path('modelos/transformer_plano_4/e5_small/best_checkpoint.pt'),
    Path('modelos/transformer_plano_4/e5_small/tokenizer/tokenizer.json'),
    Path('modelos/qwen3_06b_lora_acoso_amenaza_4/epoch_adapters/epoch_03/adapter_model.safetensors'),
]
missing_deployment = [path for path in DEPLOYMENT_REQUIRED if not (ROOT / path).is_file()]
if missing_deployment and sys.platform == 'win32' and DRIVE_BUNDLE.is_dir():
    common = ['powershell', '-NoProfile', '-ExecutionPolicy', 'Bypass', '-File', str(RECOVERY_SCRIPT), '-Workspace', str(ROOT), '-DriveBundle', str(DRIVE_BUNDLE)]
    recovery = subprocess.run([*common, '-DeploymentOnly'], text=True, capture_output=True)
    if recovery.returncode != 0: raise RuntimeError('Falló la recuperación para 05:\n' + (recovery.stderr or recovery.stdout))
    print(recovery.stdout.strip())

sync_status = pd.DataFrame([
    {'cuaderno': name, 'antes': states_before[name], 'después': _sync_state(path)}
    for name, path in COLAB_RESULT_PATHS.items()
])
display(Markdown('### Verificación Drive → workspace'))
display(sync_status)

from scripts_auxiliares import entrenar_qwen_acoso_amenaza as q4
from scripts_auxiliares import entrenar_transformers_planos_4 as t4
from scripts_auxiliares import experimentos_jerarquicos_4 as h4
from scripts_auxiliares import experimentos_jerarquicos_clasicos_4 as c4
from scripts_auxiliares import experimentos_qwen_jerarquico_4 as qh
from scripts_auxiliares import registro_modelos_produccion_4 as deploy4
# Un kernel reutilizado conserva módulos anteriores aunque el archivo haya cambiado.
importlib.invalidate_caches()
q4 = importlib.reload(q4)
t4 = importlib.reload(t4)
h4 = importlib.reload(h4)
c4 = importlib.reload(c4)
qh = importlib.reload(qh)
deploy4 = importlib.reload(deploy4)
EXPECTED_DATASET_SHA256 = 'df2ac01183271e44b6dcfb9cb4850bd6b1ef1cd11d9fc51c881be944670ef20f'
OUTPUT_DIR = ROOT / 'resultados' / 'metricas' / 'comparacion_final_4'
FIGURE_DIR = ROOT / 'resultados' / 'figuras' / 'comparacion_final_4'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

## 1. Descubrimiento y control del dataset

In [ ]:
# Defensa adicional para poder reejecutar esta celda después de actualizar el código.
if not all(hasattr(q4, name) for name in ('OPERATIONAL_SELECTION_PATH', 'OPERATIONAL_TEST_PATH', 'load_operational_evaluation')):
    import importlib
    importlib.invalidate_caches()
    q4 = importlib.reload(q4)
missing_api = [name for name in ('OPERATIONAL_SELECTION_PATH', 'OPERATIONAL_TEST_PATH', 'load_operational_evaluation') if not hasattr(q4, name)]
if missing_api:
    raise RuntimeError(f'Código auxiliar desactualizado en {q4.__file__}; faltan {missing_api}. Sincronice scripts_auxiliares y reejecute desde el inicio.')

frames = []
hashes = []
missing = []

def add_table(path, family, structure, dataset_hash, exclude_model_keys=()):
    path = Path(path)
    if not path.exists():
        missing.append(str(path.relative_to(ROOT)))
        return
    table = pd.read_csv(path)
    if exclude_model_keys and 'model_key' in table.columns:
        table = table.loc[~table['model_key'].isin(exclude_model_keys)].copy()
    table['familia'] = family
    table['estructura'] = structure
    table['fuente'] = str(path.relative_to(ROOT))
    frames.append(table)
    hashes.append((str(path.relative_to(ROOT)), dataset_hash))

if q4.OPERATIONAL_SELECTION_PATH.exists() and q4.OPERATIONAL_TEST_PATH.exists():
    evaluation = q4.load_operational_evaluation(load_scores=False, require_test=True)
    rows = []
    for split, values in evaluation['metrics'].items():
        rows.append({'model_key':'qwen4_flat','modelo':f"Qwen3-0.6B LoRA plano · época operativa {evaluation['selected_epoch']}",'split':split, **{k:v for k,v in values.items() if k != 'category_recall'}})
    table = pd.DataFrame(rows)
    table['familia']='Qwen'; table['estructura']='plano'; table['checkpoint_epoch']=evaluation['selected_epoch']; table['fuente']=str(q4.OPERATIONAL_SELECTION_PATH.relative_to(ROOT))
    frames.append(table); hashes.append((str(q4.OPERATIONAL_SELECTION_PATH.relative_to(ROOT)), evaluation['dataset_sha256']))
else:
    missing.extend(str(path.relative_to(ROOT)) for path in (q4.OPERATIONAL_SELECTION_PATH, q4.OPERATIONAL_TEST_PATH) if not path.exists())

if t4.RESULT_PATH.exists():
    result=json.loads(t4.RESULT_PATH.read_text(encoding='utf-8'))
    add_table(t4.METRICS_DIR/'comparacion.csv','Transformer','plano',result['dataset']['sha256'])
else: missing.append(str(t4.RESULT_PATH.relative_to(ROOT)))

if c4.RESULT_PATH.exists():
    result=json.loads(c4.RESULT_PATH.read_text(encoding='utf-8'))
    add_table(c4.COMMON_4A1_PATH,'Clásico','plano/cascada/jerárquico',result['dataset']['balanced_dataset_sha256'])
else: missing.append(str(c4.RESULT_PATH.relative_to(ROOT)))

for key, structure in [(h4.CASCADE_EXTRA_SAFE_KEY,'cascada'),(h4.JOINT_KEY,'multitarea')]:
    result_path=h4.result_path(key)
    if result_path.exists():
        result=json.loads(result_path.read_text(encoding='utf-8'))
        add_table(h4._experiment_paths(key)['comparison'],'Transformer',structure,result['dataset']['sha256'])
    else: missing.append(str(result_path.relative_to(ROOT)))

if qh.RESULT_PATH.exists():
    result=json.loads(qh.RESULT_PATH.read_text(encoding='utf-8'))
    operational_epoch = q4.load_operational_evaluation(load_scores=False, require_test=True)['selected_epoch']
    if result.get('qwen_flat_reference', {}).get('selected_epoch') != operational_epoch:
        result = qh.refresh_comparison_from_existing(bootstrap_replicates=1_000)
        print(f'Comparación 04_206 actualizada sin reentrenar contra la época operativa {operational_epoch}.')
    add_table(qh.METRICS_DIR/'comparacion.csv','Qwen','cascada/multitarea',result['dataset']['sha256'], exclude_model_keys=('qwen4_flat',))
else: missing.append(str(qh.RESULT_PATH.relative_to(ROOT)))

bad_hashes=[item for item in hashes if item[1] != EXPECTED_DATASET_SHA256]
if bad_hashes: raise ValueError(f'Resultados con otro dataset: {bad_hashes}')
display(pd.DataFrame(hashes,columns=['fuente','dataset_sha256']))
display(Markdown('**Pendientes:** ' + (', '.join(missing) if missing else 'ninguno')))

## 2. Tabla comparable

Se eliminan referencias duplicadas que algunos cuadernos incluyen como control. La selección global usa PR-AUC macro de validation; test sólo describe el modelo seleccionado y las demás alternativas.

In [ ]:
if not frames: raise RuntimeError('Todavía no hay experimentos terminados.')
comparison=pd.concat(frames,ignore_index=True,sort=False)
required=['modelo','split','damage_pr_auc_macro','damage_f1_macro','any_damage_recall','missed_damage_as_safe']
comparison=comparison.dropna(subset=[c for c in required if c in comparison]).copy()
comparison=comparison.drop_duplicates(subset=['modelo','split'],keep='last')
comparison.to_csv(OUTPUT_DIR/'comparacion_todos_modelos_4.csv',index=False)
deployment_registry = deploy4.build_registry()
validation=comparison.loc[comparison['split'].eq('validation')].sort_values(['damage_pr_auc_macro','damage_f1_macro'],ascending=False)
winner=str(validation.iloc[0]['modelo'])
test=comparison.loc[comparison['split'].eq('test')].sort_values('damage_pr_auc_macro',ascending=False)
display(Markdown(f'### Ganador por validation: **{winner}**'))
display(validation[['modelo','familia','estructura','damage_pr_auc_macro','damage_f1_macro','any_damage_recall']])
display(test[['modelo','familia','estructura','damage_pr_auc_macro','damage_f1_macro','any_damage_recall','missed_damage_as_safe']])
display(Markdown('### Modelos publicados para el cuaderno 05'))
display(pd.DataFrame([{'tipo': slot, 'modelo': value['label'], 'PR-AUC validation': value['validation_damage_pr_auc_macro']} for slot, value in deployment_registry['models'].items()]))
print('Registro:', deploy4.REGISTRY_PATH.relative_to(ROOT))

In [ ]:
plot=test.set_index('modelo')[['damage_pr_auc_macro','damage_f1_macro','any_damage_recall']]
ax=plot.plot.bar(figsize=(14,6))
ax.set_ylim(0,1)
ax.set_title('Comparación sobre el mismo test 4:1')
ax.grid(axis='y',alpha=.25)
plt.tight_layout()
plt.savefig(FIGURE_DIR/'comparacion_todos_modelos_4.png',dpi=180,bbox_inches='tight')
plt.show()

## Interpretación

El ranking global por validation no sustituye los contrastes pareados de cada cuaderno jerárquico. Para producción también deben revisarse recall mínimo por categoría, falsos negativos, tasa de revisión y los intervalos por video. Ningún modelo se considera autónomo sin gold standard humano independiente y piloto prospectivo.